# Week 7: A Minimal RAG Pipeline, Step by Step

<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week7_rag_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Retrieval-augmented generation (RAG)** answers a question in two moves: find the passages that contain the answer, then let a language model read them and reply. This notebook builds every piece from scratch, shows what each one does on real examples, and then measures the whole pipeline.

```
 documents ──► chunks ──► index (BM25 / embeddings)
                               │
 question ─────────────────────┴──► top k chunks ──► prompt ──► LLM ──► answer ──► score
```

| | |
|---|---|
| Paper | [Lewis et al., 2020](https://arxiv.org/abs/2005.11401): retrieval beats closed-book on knowledge questions, and swapping the index updates what the model knows |
| Documents | 60 pages about **fictional** products, so the model cannot answer from memory |
| Models | Qwen2.5-1.5B-Instruct and 3B-Instruct; embeddings from bge-small-en-v1.5 |
| Plan and hypotheses | Fixed before this run: [`notes/week_7/README.md`](https://github.com/jorshimayor/AI-Engineering-Flow_Research/blob/main/notes/week_7/README.md) |
| Code | [`src/rag_corpus.py`](https://github.com/jorshimayor/AI-Engineering-Flow_Research/blob/main/src/rag_corpus.py) · [`src/rag.py`](https://github.com/jorshimayor/AI-Engineering-Flow_Research/blob/main/src/rag.py) · [`src/run_rag_bench.py`](https://github.com/jorshimayor/AI-Engineering-Flow_Research/blob/main/src/run_rag_bench.py) |

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. About 30 minutes. Results save to Google Drive after every batch, so a re-run after a disconnect continues where it stopped. At the end the notebook saves a copy of itself **with all outputs**, so the results, tables and graphs stay readable on GitHub.

## 0. Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
REPO = "https://github.com/jorshimayor/AI-Engineering-Flow_Research.git"
BRANCH = "main"
import os
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
CODE = os.path.join(BASE, "AI-Engineering-Flow_Research")
if not os.path.exists(CODE):
    !git clone --depth 1 --branch {BRANCH} {REPO} {CODE}
%cd {CODE}
!git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
!pip install -q "transformers==4.49.0" accelerate
!git log --oneline -1

In [ ]:
USE_DRIVE = True
RESULTS = "results/week7"
if USE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/AI-Engineering/week7"
    os.makedirs(target, exist_ok=True)
    if os.path.isdir(RESULTS) and not os.path.islink(RESULTS):   # keep files the repo already has here
        for f in os.listdir(RESULTS):
            if os.path.isfile(os.path.join(RESULTS, f)) and not os.path.exists(os.path.join(target, f)):
                shutil.copy(os.path.join(RESULTS, f), target)
        shutil.rmtree(RESULTS)
    os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
    if not os.path.islink(RESULTS):
        os.symlink(target, RESULTS)
os.makedirs(RESULTS, exist_ok=True)
print("results go to:", os.path.realpath(RESULTS))

In [ ]:
import sys, importlib, inspect, json
from pathlib import Path
sys.path.insert(0, "src")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Code, Image, Markdown, display
import rag_corpus, rag, run_rag_bench, figures
for m in (rag_corpus, rag, run_rag_bench, figures): importlib.reload(m)
assert (rag_corpus.CORPUS_VERSION, rag.RAG_VERSION) == ("1.0", "1.0"), "stale code: Runtime → Disconnect and delete runtime, then Run all"
pd.set_option("display.max_colwidth", 120)

def show(fn):
    """Print a function's source from the repo, so the notebook shows the exact code that runs."""
    display(Code(inspect.getsource(fn), language="python"))

## 1. The documents

Every product, brand, town and number is invented, so the closed-book model has nothing to recall. The four models in a family share almost all their wording and differ in the numbers. That is the trap for retrieval: the C-100, C-200, C-300 and C-400 pages look nearly the same.

In [ ]:
DOCS = rag_corpus.build_corpus()
print(len(DOCS), "pages,", sum(d["split"] == "test" for d in DOCS), "of them about test products")
pd.DataFrame([{"doc_id": d["doc_id"], "name": d["name"], "type": d["product_type"], "split": d["split"],
               "words": len(d["text"].split()), **d["values"]} for d in DOCS]).head(8)

In [ ]:
example = next(d for d in DOCS if d["doc_id"] == "C-200")
print(example["text"])

Each page states 7 facts. A question asks for one of them. Some are worded differently from the page ("first come out" vs "first released"):

In [ ]:
QS = rag_corpus.make_questions("test", 200, docs=DOCS)
pd.DataFrame(QS)[["qid", "doc_id", "attribute", "question", "gold", "sibling_values"]].head(8)

In [ ]:
pd.Series([q["attribute"] for q in QS]).value_counts().rename("questions per fact").to_frame().T

## 2. Chunking

A retriever works on **chunks**, not whole pages. Here: windows of 64 words that overlap by 16, so a sentence cut at one boundary appears whole in the next chunk.

In [ ]:
show(rag.chunk_doc)

In [ ]:
for c in rag.chunk_doc(example, 64, 16):
    print(f"--- {c['chunk_id']}  words {c['start']} to {c['end']}")
    print(c["text"], "\n")

Look at the last chunk: it says "the C-200" but never "Corvane C-200". A chunk can lose the context that made it findable. One cheap fix is to start every chunk with the page title (`header=True`), tested in the retrieval sweep below:

In [ ]:
print(rag.chunk_doc(example, 64, 16, header=True)[-1]["text"])

In [ ]:
sizes = {f"{w} words": rag.chunk_corpus(DOCS, w, w // 4) for w in (32, 64, 128)}
sizes["whole page"] = rag.chunk_corpus(DOCS, None)
pd.DataFrame([{"chunking": k, "chunks": len(v), "mean words": np.mean([len(c["text"].split()) for c in v])}
              for k, v in sizes.items()])

## 3. Retrieval

**BM25** scores a chunk by the question words it contains, weighting rare words (such as `c-200`) more and long chunks less. It is about 30 lines, written from scratch:

In [ ]:
show(rag.BM25)

**Dense retrieval** turns each chunk and the question into a 384-number vector with an embedding model. Similar meaning gives similar vectors, and the score is their cosine similarity.

In [ ]:
show(rag.DenseEmbedder.encode)
EMB = rag.DenseEmbedder()
CHUNKS = rag.chunk_corpus(DOCS)
INDEX = rag.Index(CHUNKS, EMB)
print(len(CHUNKS), "chunks, vectors of shape", INDEX.vecs.shape)

**Hybrid** combines the two rankings with reciprocal rank fusion: each chunk gets 1 / (60 + rank) from each retriever.

Here is one question run through all three. `answer here?` marks chunks from the right page that contain the answer:

In [ ]:
def top(q, method, k=5):
    hits = INDEX.search(q["question"], k, method)
    return pd.DataFrame([{"rank": h["rank"], "chunk": h["chunk_id"], "score": round(h["score"], 3),
                          "answer here?": "✓" if h["doc_id"] == q["doc_id"] and q["gold_text"] in h["text"] else "",
                          "text": h["text"][:110] + "…"} for h in hits])

q = QS[0]
print(q["question"], "→ gold:", q["gold_text"], "on page", q["doc_id"])
for method in ("bm25", "dense", "hybrid"):
    display(Markdown(f"**{method}**")); display(top(q, method))

### How close are sibling pages in embedding space?

If the four pages in a family have near-identical vectors, the dense retriever has little to tell them apart. The heatmap compares one family's chunks with each other and with a question about one product.

In [ ]:
fam = [c for c in CHUNKS if c["doc_id"].startswith(q["doc_id"][0])]
qv = EMB.encode([q["question"]], query=True)
V = np.vstack([qv, INDEX.vecs[[CHUNKS.index(c) for c in fam]]])
sims = V @ V.T
labels = ["question"] + [c["chunk_id"] for c in fam]
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sims, cmap="Blues", vmin=sims.min(), vmax=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sims[i, j]:.2f}", ha="center", va="center", fontsize=6, color="white" if sims[i, j] > 0.9 else "black")
ax.set_title(f"Cosine similarity: {q['doc_id'][0]}-series chunks and the question about {q['doc_id']}")
plt.colorbar(im, fraction=0.046); plt.show()

In [ ]:
scores = INDEX.scores(q["question"], "dense")
order = np.argsort(-scores)[:20]
cols = ["#1baf7a" if CHUNKS[i]["doc_id"] == q["doc_id"] else ("#eb6834" if CHUNKS[i]["doc_id"][0] == q["doc_id"][0] else "#9b9a95") for i in order]
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(range(20), scores[order], color=cols)
ax.set_xticks(range(20)); ax.set_xticklabels([CHUNKS[i]["chunk_id"] for i in order], rotation=90, fontsize=8)
ax.set_ylim(scores[order].min() - 0.02, scores[order].max() + 0.01); ax.set_ylabel("cosine similarity")
ax.set_title("Top 20 dense scores. Green: the right page. Orange: a sibling page. Grey: other families")
plt.show()

## 4. Retrieval on its own: chunk size, title in every chunk, method

Before any LLM, measure retrieval directly. **hit@k** = share of the 200 test questions where a chunk from the right page that contains the answer is in the top k. This needs only the embedding model, so it takes about a minute.

In [ ]:
sweep = run_rag_bench.retrieval_sweep(EMB, Path(RESULTS) / "retrieval.json")

In [ ]:
sw = pd.DataFrame(sweep)
sw.pivot_table(index=["chunk_words", "header"], columns="method", values="hit@3").style.format("{:.0%}").background_gradient(cmap="Blues", vmin=0, vmax=1)

In [ ]:
figures.OUT = Path(RESULTS) / "figures"; figures.RES = Path("results")
figures.week7_retrieval_sweep()
display(Image(str(figures.OUT / "week7_retrieval_sweep.png")))

The main pipeline stays at 64-word chunks, no title, dense, k = 3, as fixed in the plan. The sweep is exploratory: it shows what the fixed choice costs or gains, it does not pick the configuration after seeing the test questions.

## 5. Generation: from chunks to a prompt to an answer

The retrieved chunks go into the prompt, best first, numbered. The model is told to reply with the bare answer, or `unknown` if the context does not say.

In [ ]:
show(rag.messages)

In [ ]:
pipe = run_rag_bench.Pipeline("test", 200, EMB)
passages, hits = pipe.passages("dense_k3", q)
for m in rag.messages(q["question"], passages):
    print(f"── {m['role']} ──\n{m['content']}\n")

Scoring reads the reply as a number (or one named colour or town) and compares it with the gold value. It also flags replies that give a **sibling product's** value, the typical distractor error:

In [ ]:
show(rag.parse_answer)
for reply in ["550 W", "The G-300 is rated at 550 W.", "505", "unknown"]:
    print(f"{reply!r:35} → {rag.parse_answer(reply, q)}")

## 6. Checks before the long run

The model-free tests first: corpus, chunking, BM25, scoring, and a full run with a fake model. Then the **dev gate** from the plan: 24 dev questions (different brands from the test set) on the 1.5B model. If the oracle scores below 50% here, stop: the test run would be meaningless.

In [ ]:
!python tests/test_week7.py 2>&1 | grep -E '^ok|passed|Error|assert'

In [ ]:
from run_prompt_bench import BatchLLM
llm = BatchLLM("Qwen/Qwen2.5-1.5B-Instruct")
dev_path = Path(RESULTS) / "dev" / "Qwen2.5-1.5B-Instruct.jsonl"
run_rag_bench.run("Qwen/Qwen2.5-1.5B-Instruct", ["closed", "dense_k3", "oracle"], "dev", 24, dev_path, 24, llm=llm, embedder=EMB)
dev = pd.DataFrame(run_rag_bench.table([dev_path]))
display(dev[["condition", "n", "accuracy", "abstained", "answer_retrieved"]])
oracle_dev = dev.set_index("condition").loc["oracle", "accuracy"]
assert oracle_dev >= 0.5, f"dev gate failed: oracle {oracle_dev:.0%} < 50%. Stop and record why before the test run."
print(f"dev gate passed: oracle {oracle_dev:.0%}")

A few dev answers, to see what the model actually writes:

In [ ]:
pd.read_json(dev_path, lines=True)[["condition", "question", "gold", "output", "correct"]].sample(8, random_state=0)

## 7. The test run

Both models, all 9 conditions, the same 200 questions. Each model is loaded once.

In [ ]:
pd.DataFrame([{"condition": k, "retrieval": v[0] or "none", "k": v[1], "corpus edition": v[2]} for k, v in run_rag_bench.CONDITIONS.items()])

In [ ]:
import gc
MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
PATHS = []
for model in MODELS:
    if llm is None or llm.name != model:
        llm = None; gc.collect(); torch.cuda.empty_cache()
        llm = BatchLLM(model)
    PATHS.append(run_rag_bench.run(model, list(run_rag_bench.CONDITIONS), "test", 200,
                                   Path(RESULTS) / f"{model.split('/')[-1]}.jsonl", 16, llm=llm, pipe=pipe))

## 8. Results

The full printout first (it is also what the notes are written from), then the same numbers as tables.

In [ ]:
RES = run_rag_bench.summary(PATHS)

In [ ]:
t = pd.DataFrame(RES["_table"])
t["95% CI"] = t.apply(lambda r: f"{r.ci_lo:.0%} to {r.ci_hi:.0%}", axis=1)
(t[["model", "condition", "accuracy", "95% CI", "abstained", "sibling_answer", "answer_retrieved", "tokens_in"]]
 .style.format({"accuracy": "{:.1%}", "abstained": "{:.1%}", "sibling_answer": "{:.1%}", "answer_retrieved": "{:.1%}", "tokens_in": "{:.0f}"}, na_rep="")
 .bar(subset=["accuracy"], color="#9ec5f0", vmin=0, vmax=1))

### Confirmatory: H1 (RAG vs closed-book) and H2 (RAG vs oracle)

In [ ]:
rows = []
for model, r in RES.items():
    if model.startswith("_"): continue
    if "h1" in r:
        h = r["h1"]; rows.append(dict(model=model, hypothesis="H1: dense_k3 − closed", effect=f"{h['effect']*100:+.1f}",
            CI=f"{h['lo']*100:+.1f} to {h['hi']*100:+.1f}", discordant=h["discordant"], holm_p=f"{h['p_holm']:.2g}",
            verdict="supported" if h["supported"] else "not supported"))
    if "h2" in r:
        h = r["h2"]; rows.append(dict(model=model, hypothesis="H2: dense_k3 − oracle (±5)", effect=f"{h['effect']*100:+.1f}",
            CI=f"{h['lo']*100:+.1f} to {h['hi']*100:+.1f}", discordant=h["discordant"], holm_p="", verdict=h["verdict"]))
pd.DataFrame(rows)

### Figures

In [ ]:
for fn in (figures.week7_accuracy_by_condition, figures.week7_topk, figures.week7_failures):
    fn()
for name in ("week7_accuracy_by_condition", "week7_topk", "week7_failures"):
    display(Image(str(figures.OUT / f"{name}.png")))

## 9. Looking at the failures

Numbers say how often; examples say why. Below: questions the main pipeline got wrong, with what was retrieved and what the model said.

In [ ]:
def explain(r):
    print(f"Q{r['qid']}: {r['question']}\n   gold {r['gold']!r}, model said {r['output']!r}"
          f"  (sibling value: {r['sibling']}, answer chunk rank: {r['gold_rank']})")
    for cid in r["retrieved"]:
        c = next(x for x in pipe.index(1).chunks if x["chunk_id"] == cid)
        print(f"     [{cid}] {c['text'][:150]}…")
    print()

big = PATHS[-1]
for kind in ("miss", "sibling"):
    display(Markdown(f"**{kind}**: " + {"miss": "the answer chunk was not retrieved", "sibling": "the model gave another product's value"}[kind]))
    for r in run_rag_bench.examples(big, kind=kind, n=3):
        explain(r)

Accuracy by fact, main pipeline:

In [ ]:
pd.DataFrame({m: {a: f"{c}/{n}" for a, (c, n) in r["failures"]["accuracy_by_attribute"].items()}
              for m, r in RES.items() if not m.startswith("_") and "failures" in r})

### The index swap (Lewis et al. claim 2)

Same questions, same model, but the index now holds edition 2 of every page. Did the answers follow the new documents?

In [ ]:
swap = pd.read_json(big, lines=True)
swap = swap[swap.condition == "swap_dense_k3"]
pd.DataFrame({"correct (new value)": [swap.correct.mean()], "stale (old value)": [swap.stale.mean()],
              "said unknown": [swap.abstained.mean()]}).style.format("{:.1%}")

## 10. Save and download

The zip holds the raw results, the retrieval sweep, the figures and a copy of this notebook **with its outputs**. Unzip it into `results/` in the repo.

In [ ]:
try:   # save this notebook with every output, so the results and graphs are readable on GitHub
    from google.colab import _message
    nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    with open(os.path.join(RESULTS, "week7_rag_colab_executed.ipynb"), "w") as f:
        json.dump(nb, f)
    print("saved the executed notebook")
except Exception as e:
    print("could not save the executed notebook:", e, "\nUse File → Download → Download .ipynb instead.")

In [ ]:
zip_path = os.path.join(BASE, "week7_results.zip")
!cd results && zip -qr {zip_path} week7/
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Download", zip_path, "from the output panel")